In [1]:
### imports
import numpy as np
from numpy import ndarray
import pandas as pd
from typing import Tuple
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(42)

### mandala setup
from mandala.all import *
set_logging_level('warning')
storage = Storage(in_memory=True)

### experimental primitives
@op(storage)
def generate_data() -> Tuple[ndarray, ndarray]:
    print('generating data...')
    X, y = make_classification(n_samples=1000, class_sep=0.75)
    return X, y

@op(storage)
def preprocess_data(X, n_components=2) -> ndarray:
    print('preprocessing data...')
    return PCA(n_components=n_components).fit_transform(X)

@op(storage)
def fit_regression(X, y, C) -> LogisticRegression:
    print('fitting regression...')
    return LogisticRegression(C=C).fit(X, y)

@op(storage)
def evaluate_model(model, X, y) -> float:
    print('evaluating model...')
    return accuracy_score(y, model.predict(X))

In [2]:
with run(storage):
    X, y = generate_data()
    model = fit_regression(X, y, C=0.1)
    accuracy = evaluate_model(model, X, y)    
    print(accuracy)

generating data...


fitting regression...


evaluating model...


AtomRef(0.812, type=AtomType(float))


In [3]:
with run(storage):
    X, y = generate_data()
    model = fit_regression(X, y, C=0.1)
    accuracy = evaluate_model(model, X, y)    
    print(accuracy)

AtomRef(0.812, type=AtomType(float))


In [4]:
with run(storage):
    X, y = generate_data()
    for n_components in (2, 4):
        X_preprocessed = preprocess_data(X, n_components=n_components)
        model = fit_regression(X_preprocessed, y, C=0.1)
        accuracy = evaluate_model(model, X_preprocessed, y)    
        print(f'{n_components} components: {accuracy}') 

preprocessing data...


fitting regression...


evaluating model...


2 components: AtomRef(0.758, type=AtomType(float))


preprocessing data...


fitting regression...


evaluating model...


4 components: AtomRef(0.786, type=AtomType(float))


In [5]:
with run(storage):
    X, y = generate_data()
    for n_components in (2, 4, 8):
        X_preprocessed = preprocess_data(X, n_components=n_components)
        for C in (0.1, 1):
            model = fit_regression(X_preprocessed, y, C=C)
            accuracy = evaluate_model(model, X_preprocessed, y)    
            print(f'{n_components} components, C={C}: {accuracy}')

2 components, C=0.1: AtomRef(0.758, type=AtomType(float))


fitting regression...


evaluating model...


2 components, C=1: AtomRef(0.759, type=AtomType(float))


4 components, C=0.1: AtomRef(0.786, type=AtomType(float))


...

evaluating model...


8 components, C=0.1: AtomRef(0.8, type=AtomType(float))


fitting regression...


evaluating model...


8 components, C=1: AtomRef(0.8, type=AtomType(float))


In [6]:
with run(storage):
    X, y = generate_data()
    for n_components in (4, 8):
        X_preprocessed = preprocess_data(X, n_components=n_components)
        for C in (0.1,):
            model = fit_regression(X_preprocessed, y, C=C)
            accuracy = evaluate_model(model, X_preprocessed, y)    
            print(f'{n_components} components, C={C}: {accuracy}')

4 components, C=0.1: AtomRef(0.786, type=AtomType(float))


8 components, C=0.1: AtomRef(0.8, type=AtomType(float))


In [7]:
with query(storage) as q:
    X, y = generate_data()
    n_components = Query().named('n_components')
    X_preprocessed = preprocess_data(X, n_components=n_components)
    C = Query().named('C')
    model = fit_regression(X_preprocessed, y, C=C)
    accuracy = evaluate_model(model, X_preprocessed, y).named('accuracy')
    table = q.get_table(n_components, C, accuracy)
table

,n_components,C,accuracy
0,4,0.1,0.786
1,2,0.1,0.758
2,8,1.0,0.800
3,4,1.0,0.785
4,8,0.1,0.800
5,2,1.0,0.759


In [8]:
from sklearn.ensemble import RandomForestClassifier

@op(storage)
def train_random_forest(X, y, n_estimators=5) -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=n_estimators).fit(X, y)

@op(storage)
def get_predictions(model, X) -> ndarray:
    return model.predict(X)

@op(storage)
def get_mistake_correlation(predictions_1, predictions_2, y_true) -> float:
    return np.corrcoef(predictions_1 != y_true, predictions_2 != y_true)[0, 1] 

In [9]:
with run(storage):
    X, y = generate_data()
    for n_components_lr in (2, 4, 8): # PCA dimension for logistic regression
        for n_components_rf in (2, 4, 8): # PCA dimension for random forest
            ### logistic regression training
            X_preprocessed_lr = preprocess_data(X, n_components=n_components_lr)
            lr_models = [fit_regression(X_preprocessed_lr, y, C=C) for C in (0.1, 1)]
            lr_accuracies = [evaluate_model(model, X_preprocessed_lr, y) for model in lr_models]
            # select the best logistic regression model
            #! `unwrap()` is used to get the underlying value of a result
            best_model = lr_models[np.argmax([unwrap(acc) for acc in lr_accuracies])] 
            ### random forest training
            lr_predictions = get_predictions(best_model, X_preprocessed_lr)
            X_preprocessed_rf = preprocess_data(X, n_components=n_components_rf)
            rf_model = train_random_forest(X_preprocessed_rf, y)
            rf_predictions = get_predictions(rf_model, X_preprocessed_rf)
            # compute the correlation of mistakes!
            correlation = get_mistake_correlation(lr_predictions, rf_predictions, y)

In [10]:
import altair as alt
with run(storage):
    X, y = generate_data()
    results = []
    for n_components_lr in (2, 4, 8, 16):
        for n_components_rf in (2, 4, 8, 16):
            # logistic regression training
            X_preprocessed_lr = preprocess_data(X, n_components=n_components_lr)
            lr_models = [fit_regression(X_preprocessed_lr, y, C=C) for C in (0.1, 0.5, 1)]
            lr_accuracies = [evaluate_model(model, X_preprocessed_lr, y) for model in lr_models]
            best_model = lr_models[np.argmax([unwrap(acc) for acc in lr_accuracies])] 
            # random forest training
            lr_predictions = get_predictions(best_model, X_preprocessed_lr)
            X_preprocessed_rf = preprocess_data(X, n_components=n_components_rf)
            rf_model = train_random_forest(X_preprocessed_rf, y)
            rf_predictions = get_predictions(rf_model, X_preprocessed_rf)
            # compute the correlation of mistakes!
            correlation = get_mistake_correlation(lr_predictions, rf_predictions, y)
            results.append({'n_components_lr': n_components_lr,
                            'n_components_rf': n_components_rf,
                            'correlation': correlation})
    df = pd.DataFrame(unwrap(results))
alt.Chart(df).mark_rect().encode(x='n_components_lr:O', y='n_components_rf:O', color='correlation:Q')

fitting regression...


evaluating model...


preprocessing data...


fitting regression...


evaluating model...


...

fitting regression...


fitting regression...


evaluating model...


evaluating model...


evaluating model...


In [11]:
from typing import List
from sklearn.tree import DecisionTreeClassifier

@op(storage)
def train_tree(X, y, max_features=2, random_state=0) -> DecisionTreeClassifier:
    return DecisionTreeClassifier(random_state=random_state, max_depth=2, 
                                  max_features=max_features).fit(X, y)
    
@op(storage)
def eval_forest(trees:List[DecisionTreeClassifier], X, y) -> float:
    majority_vote = np.array([tree.predict(X) for tree in trees]).mean(axis=0) >= 0.5
    return round(accuracy_score(y_true=y, y_pred=majority_vote), 2)


In [12]:
with run(storage):
    X, y = generate_data()
    trees = [train_tree(X, y, random_state=i) for i in range(10)]
    forest_acc = eval_forest(trees, X, y)
    print(f'Random forest accuracy: {forest_acc}')

Random forest accuracy: AtomRef(0.8, type=AtomType(float))


In [13]:
with run(storage):
    X, y = generate_data()
    for n_trees in (5, 10, 20):
        trees = [train_tree(X, y, random_state=i) for i in range(n_trees)]
        forest_acc = eval_forest(trees, X, y)
        print(f'{n_trees} trees: {forest_acc}')
        print(f'Random forest accuracy: {forest_acc}')

5 trees: AtomRef(0.78, type=AtomType(float))
Random forest accuracy: AtomRef(0.78, type=AtomType(float))


10 trees: AtomRef(0.8, type=AtomType(float))
Random forest accuracy: AtomRef(0.8, type=AtomType(float))


20 trees: AtomRef(0.75, type=AtomType(float))
Random forest accuracy: AtomRef(0.75, type=AtomType(float))


In [14]:
with query(storage) as q:
    X, y = generate_data()
    tree = train_tree(X, y)
    trees = MakeList(containing=tree, at_index=0).named('trees')
    seventh_tree = trees[7].named('seventh_tree')
    forest_acc = eval_forest(trees, X, y).named('forest_acc')
    df = q.get_table(trees, forest_acc, seventh_tree)
    df['n_trees'] = df['trees'].apply(len)
    df = df.sort_values(by='n_trees')
df

,trees,forest_acc,seventh_tree,n_trees
0,"[DecisionTreeClassifier(max_depth=2, max_featu...",0.80,"DecisionTreeClassifier(max_depth=2, max_featur...",10
1,"[DecisionTreeClassifier(max_depth=2, max_featu...",0.75,"DecisionTreeClassifier(max_depth=2, max_featur...",20


In [15]:
with run(storage):
    X, y = generate_data()
    for n_trees in (5, 10, 20):
        trees = [train_tree(X, y, random_state=i) for i in range(n_trees)]
        forest_acc = eval_forest(trees, X, y)
        print(f'Random forest accuracy: {forest_acc}')

Random forest accuracy: AtomRef(0.78, type=AtomType(float))


Random forest accuracy: AtomRef(0.8, type=AtomType(float))


Random forest accuracy: AtomRef(0.75, type=AtomType(float))


In [16]:
@superop(storage)
def train_trees(X, y, n_trees) -> List[DecisionTreeClassifier]:
    print('Hi from train_trees!')
    return [train_tree(X, y, random_state=i) for i in range(n_trees)]

In [17]:
with run(storage):
    X, y = generate_data()
    for n_trees in (5, 10, 20):
        trees = train_trees(X, y, n_trees)
        forest_acc = eval_forest(trees, X, y)
        print(f'Random forest accuracy: {forest_acc}')

Hi from train_trees!


Random forest accuracy: AtomRef(0.78, type=AtomType(float))


Hi from train_trees!


Random forest accuracy: AtomRef(0.8, type=AtomType(float))


Hi from train_trees!


Random forest accuracy: AtomRef(0.75, type=AtomType(float))


In [18]:
with run(storage):
    X, y = generate_data()
    for n_trees in (5, 10, 20):
        trees = train_trees(X, y, n_trees)
        forest_acc = eval_forest(trees, X, y)
        print(f'Random forest accuracy: {forest_acc}')

Random forest accuracy: AtomRef(0.78, type=AtomType(float))


Random forest accuracy: AtomRef(0.8, type=AtomType(float))


Random forest accuracy: AtomRef(0.75, type=AtomType(float))


In [19]:
generate_data.get_table()

,output_0,output_1
0,"[[-0.6693560962502948, -1.283925687245877, -0....","[1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, ..."


In [20]:
@op(storage)
def generate_data(n_samples=CompatArg(default=1000)) -> Tuple[ndarray, ndarray]:
    print('generating data...')
    X, y = make_classification(n_samples=n_samples, class_sep=0.75)
    return X, y

In [21]:
generate_data.get_table()

,output_0,output_1,n_samples
0,"[[-0.6693560962502948, -1.283925687245877, -0....","[1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, ...",1000


In [22]:
with run(storage, lazy=True):
    for n_samples in (1_000, 2_000):
        X, y = generate_data(n_samples=n_samples)
        for n_trees in (5, 10, 20):
            trees = [train_tree(X, y, random_state=i) for i in range(n_trees)]
            forest_acc = eval_forest(trees, X, y)
            print(f'{n_trees} trees, random forest accuracy: {forest_acc}')

5 trees, random forest accuracy: AtomRef(in_memory=False, type=AtomType(float))


10 trees, random forest accuracy: AtomRef(in_memory=False, type=AtomType(float))


20 trees, random forest accuracy: AtomRef(in_memory=False, type=AtomType(float))


generating data...


5 trees, random forest accuracy: AtomRef(0.86, type=AtomType(float))


10 trees, random forest accuracy: AtomRef(0.84, type=AtomType(float))


20 trees, random forest accuracy: AtomRef(0.87, type=AtomType(float))


In [23]:
@op(storage)
def eval_tree(tree:DecisionTreeClassifier, X, y) -> float:
    return round(accuracy_score(y_true=y, y_pred=tree.predict(X)), 2)

@superop(storage, version=1)
def train_trees(X, y, n_trees) -> List[DecisionTreeClassifier]:
    print('Hi from train_trees!')
    trees = [train_tree(X, y, random_state=i) for i in range(n_trees)]
    accuracies = [eval_tree(tree, X, y) for tree in trees]
    return trees

In [24]:
with run(storage):
    X, y = generate_data()
    for n_trees in (5, 10, 20):
        trees = train_trees(X, y, n_trees)
        forest_acc = eval_forest(trees, X, y)
        print(f'Random forest accuracy: {forest_acc}')

Hi from train_trees!


Random forest accuracy: AtomRef(0.78, type=AtomType(float))


Hi from train_trees!


Random forest accuracy: AtomRef(0.8, type=AtomType(float))


Hi from train_trees!


Random forest accuracy: AtomRef(0.75, type=AtomType(float))


In [25]:
with run(storage, lazy=True):
    X, y = generate_data()
    print(f'X: {X}')
    for n_trees in (5,):
        trees = train_trees(X, y, n_trees)
        print(f'Trees: {trees}')
        forest_acc = eval_forest(trees, X, y)
        print(f'Random forest accuracy: {forest_acc}')
        print(f'Unwrapped accuracy: {unwrap(forest_acc)}')

X: AtomRef(in_memory=False, type=AtomType(ndarray))


Trees: ListRef(in_memory=False, elt_type=AtomType(DecisionTreeClassifier))


Random forest accuracy: AtomRef(in_memory=False, type=AtomType(float))


Unwrapped accuracy: 0.78


In [26]:
with run(storage, lazy=True):
    X, y = generate_data()
    for n_trees in (5,):
        trees = train_trees(X, y, n_trees)
        print(f'Trees: {trees}')
        for tree in trees:
            print(tree)
        print(trees)
        forest_acc = eval_forest(trees, X, y)
        if forest_acc > 0.5:
            print(forest_acc)

Trees: ListRef(in_memory=False, elt_type=AtomType(DecisionTreeClassifier))


AtomRef(in_memory=False, type=AtomType(DecisionTreeClassifier))
AtomRef(in_memory=False, type=AtomType(DecisionTreeClassifier))
AtomRef(in_memory=False, type=AtomType(DecisionTreeClassifier))
AtomRef(in_memory=False, type=AtomType(DecisionTreeClassifier))
AtomRef(in_memory=False, type=AtomType(DecisionTreeClassifier))
ListRef([AtomRef(in_memory=F..., elt_type=AtomType(DecisionTreeClassifier))


AtomRef(0.78, type=AtomType(float))


In [27]:
@op(storage)
def increment_large_array(large_array:np.ndarray) -> np.ndarray:
    return AsTransient(large_array + 1)

In [28]:
with run(storage):
    a = np.arange(100)
    a_plus_1 = increment_large_array(a)

In [29]:
with run(storage, lazy=True):
    a = np.arange(100)
    a_plus_1 = increment_large_array(a)
    try:
        a_plus_2 = increment_large_array(a_plus_1)
    except Exception as e:
        print(e)

The value reference AtomRef(in_memory=False, type=AtomType(ndarray)) was requested but not found in memory.


In [30]:
with run(storage, lazy=True):
    a = np.arange(100)
    with run(force=True):
        a_plus_1 = increment_large_array(a)
    a_plus_2 = increment_large_array(a_plus_1)

In [31]:
import dask

with run(storage) as context:
    X, y = generate_data()
    futures = []
    for n_components in (10, 12):
        X_preprocessed = dask.delayed(preprocess_data)(X, n_components=n_components)
        for C in (0.1, 1.0):
            model = dask.delayed(fit_regression)(X_preprocessed, y, C=C)
            accuracy = dask.delayed(evaluate_model)(model, X_preprocessed, y)    
            futures.append(accuracy)
    results = dask.compute(*futures)
results

preprocessing data...
preprocessing data...


fitting regression...


fitting regression...


fitting regression...
fitting regression...


evaluating model...


evaluating model...


evaluating model...


evaluating model...


(
    AtomRef(0.806, type=AtomType(float)),
    AtomRef(0.806, type=AtomType(float)),
    AtomRef(0.804, type=AtomType(float)),
    AtomRef(0.805, type=AtomType(float))
)

In [32]:
with run(storage) as context:
    context.init_ray()
    X, y = generate_data()
    X, y = context.ray_put(X), context.ray_put(y)
    futures = []
    for n_components in (14, 16):
        X_preprocessed = preprocess_data.remote(X, n_components=n_components)
        for C in (0.1, 1.0):
            model = fit_regression.remote(X_preprocessed, y, C=C)
            accuracy = evaluate_model.remote(model, X_preprocessed, y)    
            futures.append(accuracy)
    results = context.ray_get(futures)
results

2022-06-11 18:08:38,875	INFO services.py:1272 -- View the Ray dashboard at http://127.0.0.1:8266


(pid=762) preprocessing data...
(pid=762) fitting regression...
(pid=762) evaluating model...
(pid=755) fitting regression...
(pid=755) evaluating model...


[
    AtomRef(0.808, type=AtomType(float)),
    AtomRef(0.81, type=AtomType(float)),
    AtomRef(0.818, type=AtomType(float)),
    AtomRef(0.819, type=AtomType(float))
]

In [33]:
with query(storage) as q:
    n_components = Query().named('n_components')
    X, y = generate_data()
    X_preprocessed = preprocess_data(X, n_components=n_components)
    with q.branch():
        C = Query().named('C')
        lr_model = fit_regression(X_preprocessed, C=C)
        lr_accuracy = evaluate_model(model=lr_model, X=X_preprocessed)
        lr_table = q.get_table(n_components, C, lr_accuracy)
        print(lr_table.head())
    with q.branch():
        rf_model = train_random_forest(X_preprocessed)
        rf_table = q.get_table(n_components, rf_model)
        print(rf_table.head())

(pid=761) fitting regression...
(pid=761) evaluating model...


   n_components    C  unnamed_2
0             2  0.5      0.759
1             2  0.1      0.758
2             2  1.0      0.759
3             4  0.5      0.785
4             4  0.1      0.786


   n_components                                          unnamed_1
0             2  (DecisionTreeClassifier(max_features='auto', r...
1             4  (DecisionTreeClassifier(max_features='auto', r...
2             8  (DecisionTreeClassifier(max_features='auto', r...
3            16  (DecisionTreeClassifier(max_features='auto', r...
